In [ ]:
import json
from python.component_search.service import search_mpn_part

result = search_mpn_part("AO4407-MS", limit=10)
print(json.dumps(result, indent=2))

In [ ]:
import sqlite3
import pandas as pd  # optional but handy

path = "/root/workspace/KiCAD_MCP/part_lib/jlcpcb-components.sqlite3"
with sqlite3.connect(path) as conn:
    df = pd.read_sql_query("SELECT * FROM v_components_search;", conn)

# df now contains the entire view; drop pandas if you prefer to iterate over conn.execute(...)


In [ ]:
df.head(3)

In [ ]:
mask = (df["symbol_lib"] == 1) & df["spice_model"].notna()
filtered = df.loc[mask]


In [ ]:
len(filtered)

In [1]:
import asyncio
import json
import tempfile
from pathlib import Path

from mcp import StdioServerParameters
from mcp.client.session import ClientSession
from mcp.client.stdio import stdio_client

PROJECT_ROOT = Path("/root/workspace/KiCAD_MCP").resolve()
CONFIG_PATH = PROJECT_ROOT / "config" / "default-config.json"
NODE_ENTRY = PROJECT_ROOT / "dist" / "index.js"

server_params = StdioServerParameters(
    command="node",
    args=[str(NODE_ENTRY), "--config", str(CONFIG_PATH)],
    cwd=str(PROJECT_ROOT),
)

stdio_ctx = stdio_client(server_params)
read, write = await stdio_ctx.__aenter__()
session_ctx = ClientSession(read, write)
session = await session_ctx.__aenter__()
await session.initialize()

tmp_dir = Path(tempfile.mkdtemp(prefix="schematic_one_comp_"))
schematic_result = await session.call_tool(
    name="create_schematic",
    arguments={"projectName": "one_component_demo", "path": str(tmp_dir)},
)
if schematic_result.isError:
    print("create_schematic failed:", schematic_result.content)
else:
    schematic_path = json.loads(schematic_result.content[0].text)["file_path"]
    add_result = await session.call_tool(
        name="add_schematic_component",
        arguments={
            "schematicPath": schematic_path,
            "component": {
                "type": "AO4407-MS",
                "reference": "Q1",
                "value": "10k",
                "library": "Triode_MOS_Tube_Transistor",
                "x": 50,
                "y": 50,
            },
        },
    )
    print("add_schematic_component:", add_result)

await session_ctx.__aexit__(None, None, None)
await stdio_ctx.__aexit__(None, None, None)


add_schematic_component: meta=None content=[TextContent(type='text', text='{\n  "success": true,\n  "component": {\n    "reference": "Q1",\n    "value": "10k",\n    "libId": "Triode_MOS_Tube_Transistor:AO4407-MS",\n    "unit": 1,\n    "footprint": "Triode_MOS_Tube_Transistor:Triode_MOS_Tube_Transistor:SOP-8_L5.0-W4.0-P1.27-LS6.0-BL",\n    "position": {\n      "x": 49.53,\n      "y": 49.53,\n      "rotation": 0\n    }\n  },\n  "CurrentSchematicStates": "=== Schematic State ===\\n\\nAllocated components and pins:\\nQ1: AO4407-MS, 10k\\n  Pin 1: unspecified (S)\\n  Pin 2: unspecified (S)\\n  Pin 3: unspecified (S)\\n  Pin 4: unspecified (G)\\n  Pin 5: unspecified (D)\\n  Pin 6: unspecified (D)\\n  Pin 7: unspecified (D)\\n  Pin 8: unspecified (D)\\n"\n}', annotations=None, meta=None)] structuredContent=None isError=False


False